# baza500 — FAQ Scraper: 5 polskich sklepów meblowych

**Cel:** Zebranie 500 unikalnych par pytanie/odpowiedź z kart FAQ 5 (docelowo 20) sklepów meblowych w Polsce.

## Zbadane sklepy i struktury HTML

| Sklep | URL FAQ | Pytanie (selektor) | Odpowiedź (selektor) |
|---|---|---|---|
| **Mebligo** | `/content/9-najczesciej-zadawane-pytania-faq` | `div[id^="ac-"] button[id^="ac-trigger-"]` | następne rodzeństwo `div` w bloku akordeonu |
| **Stolar Meble** | `/faq/` | `button` w `div#content` | następny element po `button` |
| **MeblujemyDOM** | `/pl/help/faq-najczesciej-zadawane-pytania-2` | elementy z `?` w `div#content` (nie-linki) | kolejne bloki tekstowe do następnego `?` |
| **SalonMeblowy.net** | `/faq.ehtml` | `h2`, `h3` w `div#afaq` | kolejne `p`/`div` do następnego nagłówka |
| **MebleM4** | `/faq-najczesciej-zadawane-pytania-w-meblem4-pl,p39.html` | elementy pasujące do `^\d+\.\s+` | kolejne bloki tekstowe do następnego numeru |


## Instalacja zależności

In [ ]:
pip install requests beautifulsoup4 pandas rapidfuzz

## Importy i konfiguracja

In [3]:
import re
import requests
from bs4 import BeautifulSoup
import pandas as pd

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    )
}

def get_soup(url: str) -> BeautifulSoup:
    r = requests.get(url, headers=HEADERS, timeout=20)
    r.raise_for_status()
    return BeautifulSoup(r.text, "html.parser")

## 1. Mebligo.pl

**Struktura:** akordeon `div[id^="ac-N"]` → `button[id^="ac-trigger-N"]` (pytanie) + ukryty `div` z odpowiedzią (obecny w HTML mimo CSS display:none).

In [13]:
!pip install requests_html beautifulsoup4


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: C:\Users\USER\anaconda3\python.exe -m pip install --upgrade pip


In [16]:
import sys
print(sys.executable)
print(sys.path)

c:\Users\USER\AppData\Local\Python\pythoncore-3.14-64\python.exe
['c:\\Users\\USER\\AppData\\Local\\Python\\pythoncore-3.14-64\\python314.zip', 'c:\\Users\\USER\\AppData\\Local\\Python\\pythoncore-3.14-64\\DLLs', 'c:\\Users\\USER\\AppData\\Local\\Python\\pythoncore-3.14-64\\Lib', 'c:\\Users\\USER\\AppData\\Local\\Python\\pythoncore-3.14-64', '', 'C:\\Users\\USER\\AppData\\Roaming\\Python\\Python314\\site-packages', 'c:\\Users\\USER\\AppData\\Local\\Python\\pythoncore-3.14-64\\Lib\\site-packages']


In [18]:
%pip install requests beautifulsoup4


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [22]:
import requests
from bs4 import BeautifulSoup
import re


def parse_mebligo() -> list:
    url = "https://mebligo.pl/content/9-najczesciej-zadawane-pytania-faq"
    
    response = requests.get(url)
    soup = BeautifulSoup(response.text, 'html.parser')
    
    qas = []

    for block in soup.find_all("div", id=re.compile(r"^ac-\d+$")):
        btn = block.find("button", id=re.compile(r"^ac-trigger-"))
        if not btn:
            continue
        question = btn.get_text(" ", strip=True).rstrip(" +").strip()

        answer_parts = []
        for child in block.children:
            if hasattr(child, "name") and child.name and child != btn.parent:
                text = child.get_text(" ", strip=True)
                if text:
                    answer_parts.append(text)
        answer = " ".join(answer_parts).strip()

        if question and answer:
            qas.append({"shop": "Mebligo", "source_url": url,
                        "question": question, "answer": answer})
    return qas


r1 = parse_mebito()
print(f"Mebligo: {len(r1)} Q&A")
r1[:2]

Mebligo: 0 Q&A


[]

In [25]:
import requests
from bs4 import BeautifulSoup
from collections import Counter


URL = "https://mebligo.pl/content/9-najczesciej-zadawane-pytania-faq"


def get_soup(url: str) -> BeautifulSoup:
    resp = requests.get(url)
    resp.raise_for_status()
    return BeautifulSoup(resp.text, "html.parser")


def find_candidate_blocks(soup: BeautifulSoup, min_occurrences: int = 3):
    """
    Heurystyka: znajdź <div> z klasami powtarzającymi się >= min_occurrences.
    To typowo odpowiada sekcjom typu 'faq-item', 'accordion-item' etc.
    """
    class_lists = []
    for div in soup.find_all("div"):
        classes = tuple(sorted(div.get("class", [])))
        if classes:
            class_lists.append(classes)

    counts = Counter(class_lists)
    frequent_classes = {
        cls for cls, cnt in counts.items() if cnt >= min_occurrences
    }

    candidates = []
    for div in soup.find_all("div"):
        classes = tuple(sorted(div.get("class", [])))
        if classes in frequent_classes:
            candidates.append(div)

    return candidates, frequent_classes


def debug_candidates():
    soup = get_soup(URL)
    candidates, frequent_classes = find_candidate_blocks(soup)

    print("Najczęściej powtarzające się kombinacje klas DIV:")
    for cls in frequent_classes:
        print("  ", cls)

    print("\nPrzykładowe bloki (przycięty tekst):\n")
    for i, div in enumerate(candidates[:10], start=1):
        text = div.get_text(" ", strip=True)
        print(f"--- BLOCK {i} ---")
        print(text[:500])  # pierwsze 500 znaków
        print()


if __name__ == "__main__":
    debug_candidates()

Najczęściej powtarzające się kombinacje klas DIV:
   ('ac',)
   ('ets_mm_block_content',)
   ('clearfix', 'hidden-md-up', 'title')
   ('row',)
   ('ets_mm_block', 'mm_block_type_image')
   ('ets_mm_block', 'mm_block_type_category')
   ('ac-panel',)
   ('container',)
   ('col-md-4', 'wrapper')
   ('clearfix',)

Przykładowe bloki (przycięty tekst):

--- BLOCK 1 ---
Łatwe zwroty do 30 dni od zakupu Polski producent i polski dostawca Dostawa z wniesieniem tel:+48 455 455 044

--- BLOCK 2 ---
Łatwe zwroty do 30 dni od zakupu Polski producent i polski dostawca Dostawa z wniesieniem tel:+48 455 455 044

--- BLOCK 3 ---
search clear  Zaloguj się shopping_cart Koszyk 0 

--- BLOCK 4 ---
search clear  Zaloguj się shopping_cart Koszyk 0 

--- BLOCK 5 ---


--- BLOCK 6 ---
search clear

--- BLOCK 7 ---
search clear

--- BLOCK 8 ---
Menu Menu Powrót Strona główna Salon Meble do salonu Sofy i kanapy Narożniki Komody Elementy dopełniające Fotele Poduszki Pufy Sprawdź najczęściej kupowane produkty

In [26]:
import re
import requests
from bs4 import BeautifulSoup


URL = "https://mebligo.pl/content/9-najczesciej-zadawane-pytania-faq"


def get_soup(url: str) -> BeautifulSoup:
    resp = requests.get(url)
    resp.raise_for_status()
    return BeautifulSoup(resp.text, "html.parser")


def parse_mebligo() -> list:
    soup = get_soup(URL)
    qas = []

    # Znajdź wszystkie diva z id w formie ac-panel-<liczba>
    for panel in soup.find_all("div", id=re.compile(r"^ac-panel-(\d+)$")):
        panel_id = panel.get("id")
        m = re.match(r"ac-panel-(\d+)", panel_id)
        if not m:
            continue
        idx = m.group(1)

        # Spróbuj znaleźć odpowiadające pytanie po id ac-trigger-<liczba>
        trigger = soup.find(id=f"ac-trigger-{idx}")

        # Jeżeli trigger nie istnieje, spróbuj wziąć poprzedni nagłówek/rodzica
        question_text = None
        if trigger:
            question_text = trigger.get_text(" ", strip=True)
        else:
            # fallback: spróbuj znaleźć pytanie jako poprzedni nagłówek
            prev_heading = panel.find_previous(["h2", "h3", "button"])
            if prev_heading:
                question_text = prev_heading.get_text(" ", strip=True)

        answer_text = panel.get_text(" ", strip=True)

        if question_text and answer_text:
            qas.append(
                {
                    "shop": "Mebligo",
                    "source_url": URL,
                    "question": question_text,
                    "answer": answer_text,
                }
            )

    return qas


if __name__ == "__main__":
    r1 = parse_mebligo()
    print(f"Mebligo: {len(r1)} Q&A")
    for qa in r1[:5]:
        print("Q:", qa["question"])
        print("A:", qa["answer"])
        print("-" * 40)

Mebligo: 0 Q&A


In [28]:
from requests_html import HTMLSession
from bs4 import BeautifulSoup
import re


URL = "https://mebligo.pl/content/9-najczesciej-zadawane-pytania-faq"


def get_soup(url: str) -> BeautifulSoup:
    session = HTMLSession()
    resp = session.get(url)
    html_text = resp.html.html  # pełen HTML z requests_html
    session.close()
    return BeautifulSoup(html_text, "html.parser")


def parse_mebligo() -> list:
    soup = get_soup(URL)
    qas = []

    for panel in soup.find_all("div", id=re.compile(r"^ac-panel-(\d+)$")):
        panel_id = panel.get("id")
        m = re.match(r"ac-panel-(\d+)", panel_id)
        if not m:
            continue
        idx = m.group(1)

        trigger = soup.find(id=f"ac-trigger-{idx}")

        question_text = None
        if trigger:
            question_text = trigger.get_text(" ", strip=True)
        else:
            prev_heading = panel.find_previous(["h2", "h3", "button"])
            if prev_heading:
                question_text = prev_heading.get_text(" ", strip=True)

        answer_text = panel.get_text(" ", strip=True)

        if question_text and answer_text:
            qas.append(
                {
                    "shop": "Mebligo",
                    "source_url": URL,
                    "question": question_text,
                    "answer": answer_text,
                }
            )

    return qas


r1 = parse_mebligo()
print(f"Mebligo: {len(r1)} Q&A")
r1[:5]

ModuleNotFoundError: No module named 'requests_html'

Niestety dupa! zmieniamy metodologie

In [29]:
import re
import requests
from bs4 import BeautifulSoup


URL = "https://mebligo.pl/content/9-najczesciej-zadawane-pytania-faq"


def get_soup(url: str) -> BeautifulSoup:
    resp = requests.get(url)
    resp.raise_for_status()
    return BeautifulSoup(resp.text, "html.parser")


def parse_mebligo() -> list:
    soup = get_soup(URL)
    qas = []

    # Wszystkie panele odpowiedzi: <div id="ac-panel-1">, <div id="ac-panel-2">, ...
    for panel in soup.find_all("div", id=re.compile(r"^ac-panel-(\d+)$")):
        panel_id = panel.get("id") or ""
        m = re.match(r"ac-panel-(\d+)", panel_id)
        if not m:
            continue
        idx = m.group(1)

        # Spróbuj znaleźć pytanie po id="ac-trigger-N"
        trigger = soup.find(id=f"ac-trigger-{idx}")

        question_text = None
        if trigger:
            question_text = trigger.get_text(" ", strip=True)
        else:
            # fallback: poprzedni nagłówek (np. h2/h3/button) przed panelem
            prev_heading = panel.find_previous(["h2", "h3", "button"])
            if prev_heading:
                question_text = prev_heading.get_text(" ", strip=True)

        answer_text = panel.get_text(" ", strip=True)

        if question_text and answer_text:
            qas.append(
                {
                    "shop": "Mebligo",
                    "source_url": URL,
                    "question": question_text,
                    "answer": answer_text,
                }
            )

    return qas


def debug_panels(limit: int = 3):
    soup = get_soup(URL)
    panels = soup.find_all("div", id=re.compile(r"^ac-panel-(\d+)$"))
    print(f"Znaleziono {len(panels)} paneli ac-panel-*")
    for i, panel in enumerate(panels[:limit], start=1):
        print(f"--- PANEL {i} ({panel.get('id')}) ---")
        print(panel.prettify()[:1000])
        print()


# Najpierw podejrzyj, czy w ogóle widzimy ac-panel-*
debug_panels()

# Potem uruchom właściwy parser
r1 = parse_mebligo()
print(f"Mebligo: {len(r1)} Q&A")
r1[:5]

Znaleziono 0 paneli ac-panel-*
Mebligo: 0 Q&A


[]

In [30]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from bs4 import BeautifulSoup
import re

URL = "https://mebligo.pl/content/9-najczesciej-zadawane-pytania-faq"

def get_soup_with_selenium() -> BeautifulSoup:
    options = Options()
    options.add_argument("--headless=new")
    driver = webdriver.Chrome(options=options)

    try:
        driver.get(URL)
        # tu ewentualnie: poczekaj na elementy FAQ, np. WebDriverWait(...)
        html = driver.page_source
    finally:
        driver.quit()

    return BeautifulSoup(html, "html.parser")


def parse_mebligo() -> list:
    soup = get_soup_with_selenium()
    qas = []

    # teraz SELENIUM widzi to samo co DevTools, więc istnieją ac-panel-*
    for panel in soup.find_all("div", id=re.compile(r"^ac-panel-(\d+)$")):
        panel_id = panel.get("id") or ""
        m = re.match(r"ac-panel-(\d+)", panel_id)
        if not m:
            continue
        idx = m.group(1)

        trigger = soup.find(id=f"ac-trigger-{idx}")

        question_text = None
        if trigger:
            question_text = trigger.get_text(" ", strip=True)
        else:
            prev_heading = panel.find_previous(["h2", "h3", "button"])
            if prev_heading:
                question_text = prev_heading.get_text(" ", strip=True)

        answer_text = panel.get_text(" ", strip=True)

        if question_text and answer_text:
            qas.append(
                {
                    "shop": "Mebligo",
                    "source_url": URL,
                    "question": question_text,
                    "answer": answer_text,
                }
            )

    return qas

ModuleNotFoundError: No module named 'selenium'

In [31]:
%pip install selenium

   ---------------------------------------- 0.0/9.7 MB ? eta -:--:--
   -- ------------------------------------- 0.5/9.7 MB 3.4 MB/s eta 0:00:03
   ------ --------------------------------- 1.6/9.7 MB 4.2 MB/s eta 0:00:02
   ---------- ----------------------------- 2.6/9.7 MB 4.5 MB/s eta 0:00:02
   --------------- ------------------------ 3.7/9.7 MB 4.6 MB/s eta 0:00:02
   ------------------- -------------------- 4.7/9.7 MB 4.6 MB/s eta 0:00:02
   ----------------------- ---------------- 5.8/9.7 MB 4.7 MB/s eta 0:00:01
   ---------------------------- ----------- 6.8/9.7 MB 4.7 MB/s eta 0:00:01
   ------------------------------- -------- 7.6/9.7 MB 4.7 MB/s eta 0:00:01
   ------------------------------------ --- 8.9/9.7 MB 4.7 MB/s eta 0:00:01
   ---------------------------------------- 9.7/9.7 MB 4.6 MB/s  0:00:02

   ----------------------------------------  0/12 [sortedcontainers]
   --- ------------------------------------  1/12 [websocket-client]
   --- ----------------------------

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [32]:
%pip install chromedriver-autoinstaller

Note: you may need to restart the kernel to use updated packages.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [33]:
%pip install chromedriver-autoinstaller

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [34]:
import re
import time

import chromedriver_autoinstaller
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from bs4 import BeautifulSoup
import requests  # może się przydać gdzie indziej


URL = "https://mebligo.pl/content/9-najczesciej-zadawane-pytania-faq"


def get_soup_with_selenium() -> BeautifulSoup:
    # Upewnij się, że chromedriver jest zainstalowany
    chromedriver_autoinstaller.install()

    options = Options()
    options.add_argument("--headless=new")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")

    driver = webdriver.Chrome(options=options)

    try:
        driver.get(URL)

        # Prosty wait – możesz później zastąpić WebDriverWait
        time.sleep(3)

        html = driver.page_source
    finally:
        driver.quit()

    return BeautifulSoup(html, "html.parser")


def parse_mebligo() -> list:
    soup = get_soup_with_selenium()
    qas = []

    # Wszystkie panele odpowiedzi: <div id="ac-panel-1">, <div id="ac-panel-2">, ...
    panels = soup.find_all("div", id=re.compile(r"^ac-panel-(\d+)$"))
    print(f"Znaleziono {len(panels)} paneli ac-panel-*")

    for panel in panels:
        panel_id = panel.get("id") or ""
        m = re.match(r"ac-panel-(\d+)", panel_id)
        if not m:
            continue
        idx = m.group(1)

        # Pytanie – spróbujmy id="ac-trigger-<n>"
        trigger = soup.find(id=f"ac-trigger-{idx}")

        question_text = None
        if trigger:
            question_text = trigger.get_text(" ", strip=True)
        else:
            # fallback: poprzedni nagłówek lub button
            prev_heading = panel.find_previous(["h2", "h3", "button"])
            if prev_heading:
                question_text = prev_heading.get_text(" ", strip=True)

        answer_text = panel.get_text(" ", strip=True)

        if question_text and answer_text:
            qas.append(
                {
                    "shop": "Mebligo",
                    "source_url": URL,
                    "question": question_text,
                    "answer": answer_text,
                }
            )

    return qas


r1 = parse_mebligo()
print(f"Mebligo: {len(r1)} Q&A")
r1[:5]

Znaleziono 45 paneli ac-panel-*
Mebligo: 45 Q&A


[{'shop': 'Mebligo',
  'source_url': 'https://mebligo.pl/content/9-najczesciej-zadawane-pytania-faq',
  'question': 'Czy można gdzieś zobaczyć meble przed dostawą/usiąść na nich?',
  'answer': 'Niestety, nie posiadamy stacjonarnych salonów sprzedaży.'},
 {'shop': 'Mebligo',
  'source_url': 'https://mebligo.pl/content/9-najczesciej-zadawane-pytania-faq',
  'question': 'Czy jest możliwość modyfikacji mebla (zmiana wymiarów, koloru)?',
  'answer': 'Niestety, nie produkujemy mebli na wymiar. W wybranych modelach mebli można konfigurować dostępne parametry.'},
 {'shop': 'Mebligo',
  'source_url': 'https://mebligo.pl/content/9-najczesciej-zadawane-pytania-faq',
  'question': 'Gdzie znajdę instrukcję montażu?',
  'answer': 'Instrukcje montażu naszych mebli otrzymasz w smsie informującym o etapie realizacji Twojego zamówienia. Znajdziesz je też na stronie naszego sklepu, w produktach w zakładce Instrukcje.'},
 {'shop': 'Mebligo',
  'source_url': 'https://mebligo.pl/content/9-najczesciej-zadawa

In [35]:
import csv
import os

# Definicja ścieżki
BASE_DIR = r"C:\1\T4\jdszr24-grupa-4\SKRAPOWANIE\SKRAP_DED\DATA"
INPUT_CSV = os.path.join(BASE_DIR, "faq_input.csv")
OUTPUT_CSV = os.path.join(BASE_DIR, "faq_output.csv")

def save_to_csv(qas, filename):
    fieldnames = ["shop", "source_url", "question", "answer"]
    with open(filename, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(qas)

def load_from_csv(filename):
    fieldnames = ["shop", "source_url", "question", "answer"]
    qas = []
    with open(filename, "r", newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f, fieldnames=fieldnames)
        for row in reader:
            qas.append(row)
    return qas

# Tworzenie ścieżki (jeśli nie istnieje)
os.makedirs(BASE_DIR, exist_ok=True)

# Przykładowe dane (zastąp przez parse_mebligo() jeśli potrzebujesz)
r1 = [
    {"shop": "Mebligo", "source_url": "https://mebligo.pl/faq1", "question": "Jakie są czasy dostawy?", "answer": "Dostawa w 2-3 dni robocze."},
    {"shop": "Mebligo", "source_url": "https://mebligo.pl/faq2", "question": "Czy mogę zwrócić produkt?", "answer": "Tak, w ciągu 14 dni bez wyjaśnienia przyczyny."}
]

# Zapis do faq_input.csv
save_to_csv(r1, INPUT_CSV)
print(f"✓ {INPUT_CSV} utworzony: {len(r1)} Q&A")

# Odczyt z faq_input.csv i zapis do faq_output.csv
data = load_from_csv(INPUT_CSV)
save_to_csv(data, OUTPUT_CSV)
print(f"✓ {OUTPUT_CSV} utworzony: {len(data)} Q&A")

✓ C:\1\T4\jdszr24-grupa-4\SKRAPOWANIE\SKRAP_DED\DATA\faq_input.csv utworzony: 2 Q&A
✓ C:\1\T4\jdszr24-grupa-4\SKRAPOWANIE\SKRAP_DED\DATA\faq_output.csv utworzony: 3 Q&A


In [37]:
import sys
from pathlib import Path

# Ścieżka do katalogu, gdzie leży skraper_faq.py
project_dir = Path(r"C:\1\T4\jdszr24-grupa-4\SKRAPOWANIE\SKRAP_DED")
sys.path.append(str(project_dir))

print(project_dir in map(Path, map(str, sys.path)))  # dla pewności

True


In [41]:
from skraper_faq import process_urls

process_urls("faq_input.csv", output_csv="faq_output.csv")

[INFO] INPUT:  C:\1\T4\jdszr24-grupa-4\SKRAPOWANIE\SKRAP_DED\DATA\faq_input.csv
[INFO] OUTPUT: C:\1\T4\jdszr24-grupa-4\SKRAPOWANIE\SKRAP_DED\DATA\faq_output.csv

[INFO] Przetwarzam: unknown -> https://forte.com.pl
[INFO] unknown: znaleziono 0 paneli ac-panel-*
[INFO] unknown: znaleziono 0 Q&A
[WARN] unknown: brak Q&A, nic nie zapisano

[INFO] Przetwarzam: unknown -> https://szynaka.pl/pytania-i-odpowiedzi/
[INFO] unknown: znaleziono 0 paneli ac-panel-*
[INFO] unknown: znaleziono 0 Q&A
[WARN] unknown: brak Q&A, nic nie zapisano

[INFO] Przetwarzam: unknown -> https://wersal.pl
[INFO] unknown: znaleziono 0 paneli ac-panel-*
[INFO] unknown: znaleziono 0 Q&A
[WARN] unknown: brak Q&A, nic nie zapisano

[INFO] Przetwarzam: unknown -> https://adriana.com.pl
[INFO] unknown: znaleziono 0 paneli ac-panel-*
[INFO] unknown: znaleziono 0 Q&A
[WARN] unknown: brak Q&A, nic nie zapisano

[INFO] Przetwarzam: unknown -> https://meblewojcik.pl/faq-czeste-pytania/
[INFO] unknown: znaleziono 0 paneli ac-pa

KeyboardInterrupt: 

## 2. StolarMeble.pl

**Struktura:** `div#content` → `button` (pytanie) + następny element rodzeństwo (odpowiedź). Wszystkie elementy widoczne bezpośrednio w DOM.

In [ ]:
def parse_stolar() -> list:
    url = "https://stolarmeble.pl/faq/"
    soup = get_soup(url)
    qas = []

    content = soup.find("div", id="content")
    if not content:
        return qas

    for btn in content.find_all("button"):
        question = btn.get_text(" ", strip=True)
        if not question:
            continue
        answer_node = btn.find_next_sibling()
        if not answer_node:
            answer_node = btn.parent.find_next_sibling()
        answer = answer_node.get_text(" ", strip=True) if answer_node else ""

        if question and answer:
            qas.append({"shop": "Stolar Meble", "source_url": url,
                        "question": question, "answer": answer})
    return qas

r2 = parse_stolar()
print(f"Stolar: {len(r2)} Q&A")
r2[:2]

## 3. MeblujemyDOM.pl

**Struktura:** `div#content` → naprzemienne bloki tekstowe: element z `?` = pytanie, kolejne bloki do następnego `?` = odpowiedź.

In [ ]:
def parse_meblujemydom() -> list:
    url = "https://meblujemydom.pl/pl/help/faq-najczesciej-zadawane-pytania-2"
    soup = get_soup(url)
    qas = []

    content = soup.find("div", id="content")
    if not content:
        return qas

    children = [c for c in content.children
                if hasattr(c, "name") and c.name is not None]
    
    i = 0
    while i < len(children):
        node = children[i]
        text = node.get_text(" ", strip=True)
        is_toc_link = bool(node.find("a")) and re.match(r"^\d+\.", text)
        is_question = (
            "?" in text
            and not is_toc_link
            and 10 < len(text) < 300
            and not node.find("a", href=True)
        )
        if is_question:
            answer_parts = []
            j = i + 1
            while j < len(children):
                next_text = children[j].get_text(" ", strip=True)
                if "?" in next_text and len(next_text) < 300:
                    break
                if next_text:
                    answer_parts.append(next_text)
                j += 1
            answer = " ".join(answer_parts).strip()
            if text and answer:
                qas.append({"shop": "MeblujemyDOM", "source_url": url,
                            "question": text, "answer": answer})
            i = j
        else:
            i += 1
    return qas

r3 = parse_meblujemydom()
print(f"MeblujemyDOM: {len(r3)} Q&A")
r3[:2]

## 4. SalonMeblowy.net.pl

**Struktura:** `div#afaq` → `h2`/`h3` = pytanie, kolejne `p`/`div`/`generic` do następnego nagłówka = odpowiedź.

In [ ]:
def parse_salonmeblowy() -> list:
    url = "https://www.salonmeblowy.net.pl/faq.ehtml"
    soup = get_soup(url)
    qas = []

    afaq = soup.find("div", id="afaq")
    if not afaq:
        return qas

    children = [c for c in afaq.children
                if hasattr(c, "name") and c.name is not None]

    i = 0
    while i < len(children):
        node = children[i]
        if node.name in ("h2", "h3"):
            question = node.get_text(" ", strip=True)
            answer_parts = []
            j = i + 1
            while j < len(children):
                sibling = children[j]
                if sibling.name in ("h2", "h3"):
                    break
                text = sibling.get_text(" ", strip=True)
                if text:
                    answer_parts.append(text)
                j += 1
            answer = " ".join(answer_parts).strip()
            if question and answer:
                qas.append({"shop": "SalonMeblowy.net", "source_url": url,
                            "question": question, "answer": answer})
            i = j
        else:
            i += 1
    return qas

r4 = parse_salonmeblowy()
print(f"SalonMeblowy: {len(r4)} Q&A")
r4[:2]

## 5. MebleM4.pl

**Struktura:** elementy pasujące do regex `^\d+\.\s+` = pytanie, kolejne bloki tekstowe do następnego numeru = odpowiedź.

In [ ]:
def parse_meblem4() -> list:
    url = "https://meblem4.pl/faq-najczesciej-zadawane-pytania-w-meblem4-pl,p39.html"
    try:
        soup = get_soup(url)
    except:
        return []
    qas = []

    # Najszerszy kontener, w którym może znajdować się treść artykułu
    content = soup.find("div", id="box_article")
    if not content:
        return qas

    elements = content.find_all(['p', 'div', 'span'])
    
    current_q = None
    current_a = []
    
    for el in elements:
        text = el.get_text(" ", strip=True)
        if not text:
            continue
            
        # Identyfikacja pytania po numeracji (np. '1. ', '2. ')
        if re.match(r"^\d+\.\s+", text) and len(text) < 200:
            if current_q and current_a:
                qas.append({"shop": "MebleM4", "source_url": url, 
                            "question": current_q, "answer": " ".join(current_a).strip()})
            current_q = text
            current_a = []
        elif current_q:
            # Dodawanie kolejnych bloków jako odpowiedź
            current_a.append(text)
            
    if current_q and current_a:
        qas.append({"shop": "MebleM4", "source_url": url, 
                    "question": current_q, "answer": " ".join(current_a).strip()})
        
    return qas

r5 = parse_meblem4()
print(f"MebleM4: {len(r5)} Q&A")
r5[:2]

## Podsumowanie i zebranie danych do DataFrame

In [ ]:
all_qas = r1 + r2 + r3 + r4 + r5
df = pd.DataFrame(all_qas)
print(f"Zebrano łącznie: {len(df)} par Q&A z 5 sklepów.")
df.head()